# 04 — Publication figures and tables (BTLA GREmLN vs GENIE3)

Consolidated **publication-asset** notebook. It **does not** re-run GENIE3 or GREmLN and **does not**
redefine the benchmark: every value is loaded from the canonical package frozen by notebook 03 in
`results/publication_data/`. It generates, displays and exports the **six main figures** and **four
main tables**, an asset index, and runs consistency checks.

Assets are written to `results/publication_assets/{figures,tables,captions,source_data}` in multiple
formats (figures: pdf/svg/png@600dpi + source-data csv + caption txt; tables: csv/xlsx/docx/html +
caption txt). Fixed, colour-blind-safe encodings: **GREmLN = blue**, **GENIE3 = orange**,
**shared = green**; observed seed TFs and non-bona-fide (broader-regulator) entries are outlined.

## 0. Setup and input audit

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.lines import Line2D
from scipy.stats import spearmanr
from IPython.display import display, Image, Markdown

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "scripts" / "bench_utils.py").exists()), _here.parent)
sys.path.insert(0, str(_root / "scripts"))
import bench_utils as bu

PUB = bu.repo_root() / "results" / "publication_data"
ASSET = bu.repo_root() / "results" / "publication_assets"
FIG, TAB, CAP, SRC = ASSET / "figures", ASSET / "tables", ASSET / "captions", ASSET / "source_data"
for d in (FIG, TAB, CAP, SRC):
    d.mkdir(parents=True, exist_ok=True)

EXPECTED = {
 "model_benchmark_specification": ["characteristic", "GENIE3", "GREmLN", "implication_for_comparison"],
 "common_seed_universe": ["seed_gene", "in_common_seed_set", "gremln_available", "genie3_reachable"],
 "candidate_universe_audit": ["TF", "is_bona_fide_tf", "tf_class", "is_BTLA_vs_TCR_seed"],
 "primary_rankings_common_universe": ["TF", "gremln_dense_rank", "genie3_dense_rank", "is_BTLA_vs_TCR_seed", "is_bona_fide_tf"],
 "seed_inclusive_rankings": ["TF", "gremln_dense_rank", "genie3_dense_rank"],
 "top25_union_primary": ["TF", "status", "in_gremln_top25", "in_genie3_top25", "is_bona_fide_tf"],
 "crispri_primary_qc_passed": ["TF", "model", "frac_native_8hr", "ontarget_qc_pass"],
 "crispri_all_screen_sensitivity": ["TF", "model", "validation_status", "frac_native_8hr", "in_screen_8hr", "response_direction", "n_predicted_native"],
 "crispri_matched_budget_top5": ["TF", "model", "frac_top5_8hr"],
 "crispri_matched_budget_top10": ["TF", "model", "frac_top10_8hr"],
 "crispri_model_summary": ["model", "mean_frac_native_8hr", "ci_lo", "ci_hi", "usable_supportive"],
 "paperclip_candidate_evidence": ["TF", "paperclip_evidence_tier", "paperclip_reviewed"],
 "multiomics_candidate_evidence": ["TF", "independent_orthogonal_support", "derived_contextual_support"],
 "multiomics_long_evidence": ["gene", "evidence_layer", "contrast", "timepoint", "condition", "direction", "support_level"],
 "candidate_integrated_evidence": ["TF", "status", "crispri_status", "frac_native_8hr", "paperclip_evidence_tier"],
}
assert PUB.exists(), f"canonical package missing: {PUB} (run notebook 03 section 12 first)"
D, audit_rows = {}, []
for name, cols in EXPECTED.items():
    p = PUB / (name + ".csv")
    assert p.exists(), f"MISSING canonical input: {p}"
    df = pd.read_csv(p); D[name] = df
    miss = [x for x in cols if x not in df.columns]
    assert not miss, f"{name}.csv missing columns {miss}"
    audit_rows.append({"file": name + ".csv", "rows": len(df), "cols": df.shape[1], "md5": bu.md5(p)})
input_audit = pd.DataFrame(audit_rows)
print("canonical package:", PUB)
print(input_audit.to_string(index=False))

In [ ]:
# ---- stop conditions (fail fast before generating any asset) ----
spec = D["model_benchmark_specification"]; seedu = D["common_seed_universe"]
primary = D["primary_rankings_common_universe"]; au = D["candidate_universe_audit"]
crispri = D["crispri_all_screen_sensitivity"]
n_common_seed = int(seedu["in_common_seed_set"].sum())

assert n_common_seed > 0, "no common seeds"
assert int(seedu["gremln_available"].sum()) >= n_common_seed and int(seedu["genie3_reachable"].sum()) >= n_common_seed
assert str(n_common_seed) in str(spec.loc[spec.characteristic == "common_seed_universe", "GENIE3"].iloc[0]), \
    "common seed count inconsistent between specification and seed universe (models differ)"
assert not primary["is_BTLA_vs_TCR_seed"].any(), "seeds present in the primary seed-excluded ranking"
_bad = crispri[(crispri.validation_status.isin(["not_in_screen", "failed_ontarget_qc"]))
               & (crispri.frac_native_8hr.fillna(-1) == 0)]
assert len(_bad) == 0, "absent/failed-QC perturbations encoded as zero response"
assert au["is_bona_fide_tf"].notna().all() and au["tf_class"].notna().all(), "unclassified TFs in audit"
assert set(primary["TF"]).issubset(set(au["TF"])), "primary TFs missing bona-fide classification"
# model-specific target sets must be kept separate (both models represented, shared TFs appear twice)
assert set(crispri["model"]) == {"GREmLN", "GENIE3"}, "model-specific CRISPRi rows missing"
print("All input-audit stop-conditions passed. common seeds =", n_common_seed,
      "| bona-fide TFs =", int(au.is_bona_fide_tf.sum()), "of", len(au))
display(input_audit)

## 1. Common style and export functions

In [ ]:
GRE, GEN, SHARED = "#0072B2", "#E69F00", "#009E73"     # colour-blind-safe (Okabe-Ito)
NEUTRAL, SEED_EDGE, NONBONA_EDGE = "#BBBBBB", "#000000", "#D55E00"
STATUS_COLOR = {"shared": SHARED, "GREmLN_specific": GRE, "GENIE3_specific": GEN}
DIR_COLOR = {"BTLA_concordant": "#1B7837", "anti_concordant": "#762A83",
             "mixed": "#B8B8B8", "no_response": "#EEEEEE"}
PRINCIPAL = ["EGR2", "BHLHE40", "JUNB", "IRF8", "AHR", "NFIL3", "PRDM1", "RBPJ", "STAT4", "REL", "TBX21"]
EN = "\u2013"
mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 600, "font.size": 11, "axes.titlesize": 12,
    "axes.labelsize": 11, "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "svg.fonttype": "none", "pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "DejaVu Sans"})

def _fmt(v, nd=3):
    if v is None: return EN
    if isinstance(v, float) and not np.isfinite(v): return EN
    if isinstance(v, (int, np.integer)): return str(int(v))
    if isinstance(v, float): return ("%.*f" % (nd, v))
    s = str(v)
    # note: literal "None" is a valid evidence label (Table 4) and is preserved; only true nulls -> EN
    return EN if s.lower() in ("nan", "") else s

def panel_label(ax, s):
    ax.text(-0.10, 1.05, s, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")

def save_fig(fig, name, source_df, caption):
    for ext in (".pdf", ".svg", ".png"):
        fig.savefig(FIG / (name + ext), bbox_inches="tight")
    source_df.to_csv(SRC / (name + "_source_data.csv"), index=False)
    (CAP / (name + "_caption.txt")).write_text(caption.strip() + chr(10))
    print("figure saved:", name, "(pdf/svg/png600 + source_data + caption)")

def _html_cell(v, bg=None, bold=False):
    v = str(v)
    st = []
    if bg: st.append("background-color:%s" % bg)
    if bold: st.append("font-weight:bold")
    return '<td style="%s;padding:3px 9px">%s</td>' % (";".join(st), v)

def export_table(df, name, title, footnotes, nd=3, cell_bg=None, bold_cells=None, disp=None, width=980, col_chars=None, zebra=True, docx_map=None):
    # cell_bg: {(row,col): hexcolour}; bold_cells: {(row,col)}; disp: {(row,col): image-only mathtext string}
    # docx_map: {value: replacement} applied ONLY in the DOCX export (e.g. {"None": "-"} for readability);
    #           CSV/XLSX/HTML/PNG retain the explicit values.
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].map(lambda v: _fmt(v, nd))
    cell_bg = cell_bg or {}; bold_cells = bold_cells or set(); disp = disp or {}
    cols = list(out.columns)
    out.to_csv(TAB / (name + ".csv"), index=False)
    try:
        out.to_excel(TAB / (name + ".xlsx"), index=False)
    except Exception as e:
        print("xlsx skipped:", e)
    # HTML with per-cell shading / bold (text labels retained so it reads without colour)
    hrows = ["<tr>" + "".join('<th style="background:#f2f2f2;text-align:left;padding:4px 9px">%s</th>' % c
                              for c in cols) + "</tr>"]
    for i in range(len(out)):
        tds = [_html_cell(out.iloc[i][cols[j]], cell_bg.get((i, j)), (i, j) in bold_cells) for j in range(len(cols))]
        hrows.append("<tr>" + "".join(tds) + "</tr>")
    (TAB / (name + ".html")).write_text("<h3>" + title + "</h3>" + chr(10)
        + '<table style="border-collapse:collapse;font-size:13px">' + chr(10) + chr(10).join(hrows)
        + chr(10) + "</table><hr>" + "<br>".join(footnotes))
    (CAP / (name + "_caption.txt")).write_text(title + chr(10) + chr(10) + chr(10).join(footnotes) + chr(10))
    try:
        from docx import Document
        from docx.shared import RGBColor
        from docx.oxml.ns import qn
        from docx.oxml import OxmlElement
        def _shade(cell, hexc):
            tcPr = cell._tc.get_or_add_tcPr(); sh = OxmlElement("w:shd")
            sh.set(qn("w:val"), "clear"); sh.set(qn("w:fill"), hexc.lstrip("#")); tcPr.append(sh)
        doc = Document(); doc.add_heading(title, level=2)
        t = doc.add_table(rows=1, cols=out.shape[1]); t.style = "Table Grid"
        for j, cn in enumerate(cols): t.rows[0].cells[j].text = str(cn)
        for i in range(len(out)):
            cells = t.add_row().cells
            for j, cn in enumerate(cols):
                _val = str(out.iloc[i][cn])
                cells[j].text = (docx_map or {}).get(_val, _val)
                if (i, j) in cell_bg: _shade(cells[j], cell_bg[(i, j)])
                if (i, j) in bold_cells:
                    for p in cells[j].paragraphs:
                        for run in p.runs: run.bold = True
        for fn in footnotes: doc.add_paragraph(fn)
        doc.save(str(TAB / (name + ".docx")))
    except Exception as e:
        print("docx skipped:", e)
    render_table_image(out, name, title, footnotes, cell_bg=cell_bg, bold_cells=bold_cells, disp=disp, col_chars=col_chars, zebra=zebra)
    print("table saved:", name, "(csv/xlsx/docx/html + png/pdf/svg + caption)")
    display(Image(filename=str(TAB / (name + ".png")), width=width))
    return out

def show_table(out, title):
    sty = (out.style.hide(axis="index").set_caption(title).set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px"), ("caption-side", "top"), ("margin-bottom", "6px")]},
        {"selector": "th", "props": [("background-color", "#f2f2f2"), ("text-align", "left"), ("padding", "4px 9px")]},
        {"selector": "td", "props": [("padding", "3px 9px"), ("border-bottom", "1px solid #eee")]}]))
    return sty

def render_table_image(df, name, title, footnotes, cap=30, fs=9, cell_bg=None, bold_cells=None, disp=None, col_chars=None, zebra=True):
    # Publication-styled table rendered to a raster/vector image (booktabs rules, zebra rows,
    # wrapped cells, title above, footnotes below). No browser/HTML dependency.
    # cell_bg {(i,j):colour} shades a body cell (overrides zebra); bold_cells {(i,j)} bolds text;
    # disp {(i,j):str} overrides the drawn text with a verbatim (unwrapped) string, e.g. mathtext.
    # col_chars: fixed per-column wrap widths (chars) so sibling tables share identical geometry.
    import textwrap
    cell_bg = cell_bg or {}; bold_cells = bold_cells or set(); disp = disp or {}
    cols = list(df.columns); n = len(df)
    if col_chars is not None:
        wrapw = [max(9, int(w)) for w in col_chars]
    else:
        raw = [max(len(str(c)), int(df[c].astype(str).map(len).max()) if n else len(str(c))) for c in cols]
        wrapw = [min(cap, max(9, w)) for w in raw]
    tot_w = sum(wrapw)
    frac = [w / tot_w for w in wrapw]; xb = np.concatenate([[0.0], np.cumsum(frac)])
    hdr = [textwrap.fill(str(c), wrapw[j]) for j, c in enumerate(cols)]
    def _txt(i, j):
        if (i, j) in disp: return disp[(i, j)]                    # verbatim (mathtext), no wrap
        return textwrap.fill(str(df.iloc[i][cols[j]]), wrapw[j])
    body = [[_txt(i, j) for j in range(len(cols))] for i in range(n)]
    fnw = [textwrap.fill(fn, max(60, int(tot_w))) for fn in footnotes]
    def _L(row): return max(1, max(s.count(chr(10)) + 1 for s in row))
    hL = _L(hdr); rL = [_L(r) for r in body]; fL = sum(f.count(chr(10)) + 1 for f in fnw)
    top_pad = 1.9; y0 = top_pad + hL; y_end = y0 + sum(rL); total = y_end + 1.1 + fL + 0.4
    figw = min(26, max(6, tot_w * 0.132 + 0.6)); figh = max(2.2, total * 0.30)
    fig, ax = plt.subplots(figsize=(figw, figh)); ax.set_xlim(0, 1); ax.set_ylim(total, 0); ax.axis("off")
    ax.text(0.0, top_pad * 0.45, title, fontsize=fs + 3, fontweight="bold", va="center")
    ax.add_patch(Rectangle((0, top_pad), 1, hL, fc="#eef1f4", ec="none"))
    for j in range(len(cols)):
        ax.text(xb[j] + 0.004, top_pad + hL / 2, hdr[j], fontsize=fs, fontweight="bold", va="center", clip_on=False)
    for yy, lw in ((top_pad, 1.3), (y0, 1.0), (y_end, 1.3)):
        ax.plot([0, 1], [yy, yy], color="#222", lw=lw)
    cy = y0
    for i in range(n):
        h = rL[i]
        if zebra and i % 2 == 1:
            ax.add_patch(Rectangle((0, cy), 1, h, fc="#f7f8fa", ec="none"))
        for j in range(len(cols)):
            if (i, j) in cell_bg:
                ax.add_patch(Rectangle((xb[j], cy), frac[j], h, fc=cell_bg[(i, j)], ec="none"))
        for j in range(len(cols)):
            fw = "bold" if (i, j) in bold_cells else "normal"
            ax.text(xb[j] + 0.004, cy + h / 2, body[i][j], fontsize=fs, va="center", fontweight=fw, clip_on=False)
        cy += h
    fy = y_end + 0.9
    for f in fnw:
        ax.text(0.0, fy, f, fontsize=fs - 1.5, va="top", color="#333", clip_on=False)
        fy += (f.count(chr(10)) + 1) * 0.95
    for ext in (".png", ".pdf", ".svg"):
        fig.savefig(TAB / (name + ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
    print("table image saved:", name + " (.png@600dpi/.pdf/.svg)")

print("style + export helpers ready. GREmLN=blue, GENIE3=orange, shared=green.")

## 2. Figure 1 — study design and fairness framework

In [ ]:
n_panel = len(seedu); n_gre = int(seedu.gremln_available.sum()); n_gen = int(seedu.genie3_reachable.sum())
n_cs = int(seedu.in_common_seed_set.sum()); n_univ = len(au); n_bona = int(au.is_bona_fide_tf.sum())
n_prim = len(primary); n_ctx = len(D["seed_inclusive_rankings"]); union = D["top25_union_primary"]; n_union = len(union)
cr = D["crispri_all_screen_sensitivity"]
n_screen = int(cr[cr.in_screen_8hr == True].TF.nunique()); n_qc = int(D["crispri_primary_qc_passed"].TF.nunique())
fig1_src = pd.DataFrame([
    ("CD4 Perturb-seq NTC cells", ""), ("Masked CD4 GENIE3 graph", "50,000 edges"),
    ("GREmLN (masked GENIE3 prior; pre-ComBat log1p-CPM input)", ""),
    ("BTLA+TCR vs TCR panel (4 h)", "%d DEGs" % n_panel),
    ("GREmLN-available seeds", str(n_gre)), ("GENIE3-reachable seeds", str(n_gen)),
    ("Identical common seed set", str(n_cs)), ("Initial common regulator universe", str(n_univ)),
    ("Bona-fide common TF universe", str(n_bona)), ("Seed-excluded primary universe", str(n_prim)),
    ("Seed-inclusive contextual universe", str(n_ctx)), ("Top-25 union (both models)", str(n_union)),
    ("CRISPRi in screen", str(n_screen)), ("CRISPRi on-target QC passed", str(n_qc)),
    ("Paperclip + multi-omics corroboration", "annotation only"),
], columns=["step", "count"])
print("source data for Figure 1:"); display(fig1_src)

In [ ]:
def _box(ax, x, y, w, h, text, fc):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.02,rounding_size=0.04",
                                fc=fc, ec="#333333", lw=1.2))
    ax.text(x, y, text, ha="center", va="center", fontsize=8.4)
def _arw(ax, p1, p2):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>", mutation_scale=11, color="#666666", lw=1.0))

INP, UNI, VAL = "#DCE9F5", "#E8F5E9", "#FDEBD0"
fig, ax = plt.subplots(figsize=(9.6, 12)); ax.set_xlim(0, 10); ax.set_ylim(0, 15); ax.axis("off")
_box(ax, 5, 14.3, 6.4, 0.8, "CD4 Perturb-seq NTC cells", INP)
_box(ax, 5, 13.1, 6.4, 0.8, "Masked CD4 GENIE3 graph  —  50,000 edges", INP)
_box(ax, 5, 11.85, 7.6, 1.0, "GREmLN (frozen): masked GENIE3 graph as prior;\npre-ComBat log1p-CPM expression input", INP)
_box(ax, 5, 10.6, 6.4, 0.8, "BTLA+TCR vs TCR panel (4 h)  —  %d DEGs" % n_panel, INP)
_box(ax, 2.7, 9.35, 4.2, 0.8, "GREmLN-available seeds  —  %d" % n_gre, UNI)
_box(ax, 7.3, 9.35, 4.2, 0.8, "GENIE3-reachable seeds  —  %d" % n_gen, UNI)
_box(ax, 5, 8.1, 5.2, 0.8, "Identical common seed set  —  %d" % n_cs, UNI)
_box(ax, 5, 6.9, 6.8, 0.8, "Initial common regulator universe  —  %d" % n_univ, UNI)
_box(ax, 5, 5.7, 6.8, 0.8, "Bona-fide common TF universe  —  %d" % n_bona, UNI)
_box(ax, 2.7, 4.5, 4.2, 0.8, "Seed-excluded PRIMARY  —  %d" % n_prim, UNI)
_box(ax, 7.3, 4.5, 4.2, 0.8, "Seed-inclusive context  —  %d" % n_ctx, UNI)
_box(ax, 5, 3.3, 5.4, 0.8, "Top-25 union (both models)  —  %d" % n_union, UNI)
_box(ax, 5, 2.05, 7.4, 1.0, "CD4 CRISPRi functional validation\n(in screen %d; on-target QC %d)" % (n_screen, n_qc), VAL)
_box(ax, 5, 0.75, 7.4, 0.8, "Paperclip + multi-omics corroboration (annotation only)", VAL)
for a, b in [(14.3, 13.1), (13.1, 11.85), (11.85, 10.6)]: _arw(ax, (5, a - 0.42), (5, b + 0.5))
_arw(ax, (5, 10.2), (2.7, 9.75)); _arw(ax, (5, 10.2), (7.3, 9.75))
_arw(ax, (2.7, 8.95), (5, 8.5)); _arw(ax, (7.3, 8.95), (5, 8.5))
for a, b in [(8.1, 6.9), (6.9, 5.7)]: _arw(ax, (5, a - 0.42), (5, b + 0.42))
_arw(ax, (5, 5.3), (2.7, 4.9)); _arw(ax, (5, 5.3), (7.3, 4.9))
_arw(ax, (2.7, 4.1), (5, 3.72))
for a, b in [(3.3, 2.05), (2.05, 0.75)]: _arw(ax, (5, a - 0.42), (5, b + 0.5))
ax.set_title("Study design and fairness framework for the GREmLN\u2013GENIE3 benchmark", fontsize=12, pad=6)
ax.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor=INP, markersize=12, label="inputs"),
                   Line2D([0], [0], marker="s", color="w", markerfacecolor=UNI, markersize=12, label="analysis universe"),
                   Line2D([0], [0], marker="s", color="w", markerfacecolor=VAL, markersize=12, label="validation / evidence")],
          loc="upper right", fontsize=8)
save_fig(fig, "fig1_study_design", fig1_src,
    "Figure 1. Study design and fairness framework. Boxes are analysis stages coloured by role "
    "(blue inputs, green analysis universe, orange validation/evidence); numbers are exact counts of "
    "cells, seeds or TFs entering each stage. Both models are restricted to the identical common seed "
    "set and the same common TF universe; CD4 CRISPRi is the functional validator; Paperclip and "
    "multi-omics provide corroboration only, never predictive input.")
plt.show()

## 3. Figure 2 — primary model-ranking comparison

In [ ]:
prim_b = primary[primary.is_bona_fide_tf].copy()
union = D["top25_union_primary"].copy(); status_map = dict(zip(union.TF, union.status))
fig2_src = union[["TF", "status", "gremln_dense_rank", "genie3_dense_rank", "is_BTLA_vs_TCR_seed", "is_bona_fide_tf"]].copy()
print("source data for Figure 2 (top-25 union):"); display(fig2_src)

In [ ]:
fig, (axA, axB) = plt.subplots(1, 2, figsize=(12.5, 5.8), gridspec_kw={"width_ratios": [1.05, 1]})
cols = [STATUS_COLOR.get(status_map.get(t), NEUTRAL) for t in prim_b.TF]
axA.scatter(prim_b.gremln_dense_rank, prim_b.genie3_dense_rank, c=cols, s=16, alpha=0.75, edgecolor="none")
rho = spearmanr(prim_b.gremln_dense_rank, prim_b.genie3_dense_rank)[0]
axA.set_xlabel("GREmLN dense rank"); axA.set_ylabel("GENIE3 dense rank")
axA.set_title("Rank\u2013rank, bona-fide common universe (n=%d)" % len(prim_b))
axA.text(0.03, 0.97, "Spearman \u03c1 = %.2f" % rho, transform=axA.transAxes, va="top", fontsize=10)
for t in PRINCIPAL:
    r = prim_b[prim_b.TF == t]
    if len(r):
        axA.annotate(t, (r.gremln_dense_rank.iloc[0], r.genie3_dense_rank.iloc[0]), fontsize=7.5,
                     xytext=(3, 3), textcoords="offset points")
axA.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor=SHARED, markersize=8, label="shared top-25"),
                    Line2D([0], [0], marker="o", color="w", markerfacecolor=GRE, markersize=8, label="GREmLN-only top-25"),
                    Line2D([0], [0], marker="o", color="w", markerfacecolor=GEN, markersize=8, label="GENIE3-only top-25"),
                    Line2D([0], [0], marker="o", color="w", markerfacecolor=NEUTRAL, markersize=8, label="other bona-fide TF")],
           loc="lower right", fontsize=7.5)
panel_label(axA, "A")

u = union.sort_values("gremln_dense_rank").reset_index(drop=True); y = np.arange(len(u))
axB.hlines(y, u.gremln_dense_rank, u.genie3_dense_rank, color=NEUTRAL, lw=1, zorder=1)
axB.scatter(u.gremln_dense_rank, y, color=GRE, s=24, zorder=3, label="GREmLN rank")
axB.scatter(u.genie3_dense_rank, y, color=GEN, s=24, zorder=3, label="GENIE3 rank")
labs = [("\u2605 " if s else "") + ("\u25B3 " if not b else "") + t
        for t, s, b in zip(u.TF, u.is_BTLA_vs_TCR_seed, u.is_bona_fide_tf)]
axB.set_yticks(y); axB.set_yticklabels(labs, fontsize=6.8); axB.invert_yaxis()
axB.set_xlabel("dense rank (1 = highest priority)")
axB.set_title("Top-25 union, aligned ranks (n=%d)" % len(u))
axB.legend(loc="lower right", fontsize=8)
panel_label(axB, "B")
save_fig(fig, "fig2_primary_rank_comparison", fig2_src,
    "Figure 2. Primary seed-excluded TF prioritisation. (A) Rank\u2013rank scatter over the bona-fide "
    "common TF universe (n=%d); each point is a TF, coloured by top-25 membership (green shared, blue "
    "GREmLN-only, orange GENIE3-only, grey other); Spearman \u03c1 annotated; principal candidates "
    "labelled. (B) Aligned dense ranks for the top-25 union: dots joined per TF (blue GREmLN, orange "
    "GENIE3); \u2605 marks observed BTLA seed TFs, \u25B3 marks non-bona-fide broader regulators. Ranks, "
    "not raw incomparable scores, are compared." % len(prim_b))
plt.show()

## 4. Figure 3 — CRISPRi target-response benchmark

In [ ]:
cr = D["crispri_all_screen_sensitivity"].copy(); ms = D["crispri_model_summary"].set_index("model")
qcp = D["crispri_primary_qc_passed"].copy()
fig3_src = cr[["TF", "model", "in_screen_8hr", "ontarget_qc_pass", "n_predicted_native",
               "n_confirmed_native_8hr", "frac_native_8hr", "p_native_8hr", "response_direction",
               "validation_status", "frac_top5_8hr", "frac_top10_8hr"]].copy()
print("source data for Figure 3:"); display(fig3_src.head(12))

In [ ]:
fig = plt.figure(figsize=(13.5, 15)); gs = fig.add_gridspec(2, 2, height_ratios=[2.5, 1], hspace=0.16, wspace=0.28)
axA = fig.add_subplot(gs[0, :]); axB = fig.add_subplot(gs[1, 0]); axC = fig.add_subplot(gs[1, 1])

qp = qcp.copy(); qp["label"] = qp["TF"] + " (" + qp["model"].str[:3] + ")"
qp = qp.sort_values(["model", "frac_native_8hr"], ascending=[True, True]).reset_index(drop=True)
yy = np.arange(len(qp))
axA.barh(yy, qp["frac_native_8hr"].astype(float),
         color=[DIR_COLOR.get(d, "#DDDDDD") for d in qp["response_direction"]],
         edgecolor=[GRE if m == "GREmLN" else GEN for m in qp["model"]], lw=1.4)
for i, r in qp.iterrows():
    axA.text(float(r["frac_native_8hr"]) + 0.005, i, "%d/%d" % (int(r["n_confirmed_native_8hr"]), int(r["n_predicted_native"])),
             va="center", fontsize=7.5)
axA.set_yticks(yy); axA.set_yticklabels(qp["label"], fontsize=8); axA.set_ylim(-0.6, len(qp) - 0.4)
axA.set_xlabel("native target-response fraction (confirmed / predicted), Stim8hr")
axA.set_title("A  Candidate-level CRISPRi target response (on-target-QC-passed, in-screen)", loc="left", fontsize=11)
axA.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor=DIR_COLOR["BTLA_concordant"], markersize=9, label="BTLA-concordant"),
                    Line2D([0], [0], marker="s", color="w", markerfacecolor=DIR_COLOR["anti_concordant"], markersize=9, label="anti-concordant"),
                    Line2D([0], [0], marker="s", color="w", markerfacecolor=DIR_COLOR["mixed"], markersize=9, label="mixed"),
                    Line2D([0], [0], marker="s", color="w", markerfacecolor=DIR_COLOR["no_response"], markersize=9, label="no response"),
                    Line2D([0], [0], marker="s", color="w", markeredgecolor=GRE, markerfacecolor="w", markersize=9, label="GREmLN (edge)"),
                    Line2D([0], [0], marker="s", color="w", markeredgecolor=GEN, markerfacecolor="w", markersize=9, label="GENIE3 (edge)")],
           loc="lower right", fontsize=7)
absent = cr[cr.validation_status.isin(["not_in_screen", "failed_ontarget_qc"])]
txt = "Not scored (shown, not zero):\n" + "; ".join(sorted(set(absent.TF + " [" + absent.validation_status.str.replace("_", " ") + "]")))
axA.text(1.005, 0.5, txt, transform=axA.transAxes, fontsize=6.6, va="center", ha="left",
         bbox=dict(boxstyle="round", fc="#f7f7f7", ec="#cccccc"))

models = ["GREmLN", "GENIE3"]; means = [ms.loc[m, "mean_frac_native_8hr"] for m in models]
lo = [ms.loc[m, "ci_lo"] for m in models]; hi = [ms.loc[m, "ci_hi"] for m in models]
err = np.array([[means[i] - lo[i] for i in range(2)], [hi[i] - means[i] for i in range(2)]])
axB.bar(models, means, color=[GRE, GEN], width=0.6, yerr=err, capsize=6, edgecolor="#333")
axB.axhline(0, color="#888", lw=0.8)
d_mean = means[0] - means[1]
axB.set_ylabel("mean target-response fraction (QC-passed)")
axB.set_title("B  Aggregate QC-passed comparison", loc="left", fontsize=11)
axB.text(0.5, 0.94, "\u0394(GREmLN\u2212GENIE3) = %+.3f\nsuperiority margin = 0.020" % d_mean,
         transform=axB.transAxes, ha="center", va="top", fontsize=8.5)

budg = pd.DataFrame({"budget": ["native", "top5", "top10"],
    "GREmLN": [ms.loc["GREmLN", "mean_frac_native_8hr"], ms.loc["GREmLN", "mean_frac_top5_8hr"], ms.loc["GREmLN", "mean_frac_top10_8hr"]],
    "GENIE3": [ms.loc["GENIE3", "mean_frac_native_8hr"], ms.loc["GENIE3", "mean_frac_top5_8hr"], ms.loc["GENIE3", "mean_frac_top10_8hr"]]})
xx = np.arange(3); w = 0.38
axC.bar(xx - w / 2, budg["GREmLN"], w, color=GRE, label="GREmLN")
axC.bar(xx + w / 2, budg["GENIE3"], w, color=GEN, label="GENIE3")
axC.set_xticks(xx); axC.set_xticklabels(["native", "top-5", "top-10"])
axC.set_ylabel("mean target-response fraction"); axC.set_title("C  Matched target-budget sensitivity", loc="left", fontsize=11)
axC.legend(fontsize=8)
save_fig(fig, "fig3_crispri_benchmark", fig3_src,
    "Figure 3. Orthogonal CD4 CRISPRi target-response benchmark (Stim8hr). (A) Per-candidate native "
    "target-response fraction for on-target-QC-passed, in-screen TFs; bar fill encodes response "
    "direction, bar edge encodes model (blue GREmLN, orange GENIE3), and k/n denominators are printed; "
    "TFs absent from the screen or failing on-target QC are listed separately, never plotted as zero. "
    "(B) Aggregate QC-passed means with bootstrap 95% CI (5,000 resamples), zero reference and the "
    "0.020 superiority margin. (C) Matched target-budget sensitivity (native / top-5 / top-10 predicted "
    "targets per TF).")
plt.show()

## 5. Figure 4 — integrated candidate evidence map

In [ ]:
ci = D["candidate_integrated_evidence"].copy()
order = {"shared": 0, "GREmLN_specific": 1, "GENIE3_specific": 2}
ci["_o"] = ci["status"].map(order).fillna(3); ci = ci.sort_values(["_o", "gremln_rank"]).reset_index(drop=True)
fig4_src = ci.drop(columns=["_o"]).copy()
print("source data for Figure 4:"); display(fig4_src.head(12))

In [ ]:
def _supp(v): return str(v).lower() not in ("", "nan", "none", "no", "false", "0", "0.0")
TIER = {"strong": 3, "moderate": 2, "weak": 1, "none": 0}
groups = [("model", ["gremln_rank", "genie3_rank"]),
          ("functional", ["frac_native_8hr", "response_direction", "ontarget_qc_pass"]),
          ("independent", ["protein_support", "phosphosite_support", "coip_support"]),
          ("derived", ["transcript_support", "tf_activity_support", "kinase_activity_support", "bionic_support"]),
          ("literature", ["paperclip_evidence_tier"])]
colnames = [x for _, cols in groups for x in cols]
head = {"gremln_rank": "GREmLN\nrank", "genie3_rank": "GENIE3\nrank", "frac_native_8hr": "CRISPRi\nfrac",
        "response_direction": "dir", "ontarget_qc_pass": "QC", "protein_support": "prot",
        "phosphosite_support": "phos", "coip_support": "coIP", "transcript_support": "trans",
        "tf_activity_support": "TFact", "kinase_activity_support": "kin", "bionic_support": "BIONIC",
        "paperclip_evidence_tier": "lit tier"}
nrow, ncol = len(ci), len(colnames)
fig, ax = plt.subplots(figsize=(12.5, max(6, 0.32 * nrow + 2)))
ax.set_xlim(0, ncol); ax.set_ylim(0, nrow); ax.invert_yaxis(); ax.axis("off")
for r in range(nrow):
    row = ci.iloc[r]
    for jc, col in enumerate(colnames):
        v = row[col]; fc = "#FFFFFF"; txt = ""
        if col in ("gremln_rank", "genie3_rank"):
            txt = "" if pd.isna(v) else str(int(v)); fc = "#EFEFEF"
        elif col == "frac_native_8hr":
            fv = float(v) if pd.notna(v) else np.nan
            fc = plt.cm.Blues(min(0.85, 0.15 + 4 * fv)) if np.isfinite(fv) else "#F3F3F3"
            txt = EN if not np.isfinite(fv) else "%.2f" % fv
        elif col == "response_direction":
            fc = DIR_COLOR.get(str(v), "#F3F3F3"); txt = {"BTLA_concordant": "+", "anti_concordant": "\u2212", "mixed": "~", "no_response": ""}.get(str(v), "")
        elif col == "ontarget_qc_pass":
            fc = "#C8E6C9" if v == True else ("#FFCDD2" if v == False else "#F3F3F3"); txt = "\u2713" if v == True else ("\u2717" if v == False else EN)
        elif col == "paperclip_evidence_tier":
            t = TIER.get(str(v), 0); fc = plt.cm.Purples(0.15 + 0.22 * t); txt = str(v) if pd.notna(v) else EN
        else:
            on = _supp(v); fc = "#4D4D4D" if on else "#F0F0F0"; txt = ""
        ax.add_patch(Rectangle((jc, r), 1, 1, fc=fc, ec="white", lw=1.2))
        if txt:
            tc = "white" if (col == "response_direction" and str(v) == "anti_concordant") else "#111"
            ax.text(jc + 0.5, r + 0.5, txt, ha="center", va="center", fontsize=7, color=tc)
    lab = ("\u2605 " if row.get("is_BTLA_vs_TCR_seed", False) else "") + row["TF"]
    ax.text(-0.2, r + 0.5, lab, ha="right", va="center", fontsize=7.2)
    ax.add_patch(Rectangle((-0.06, r + 0.06), 0.05, 0.88, fc=STATUS_COLOR.get(row["status"], NEUTRAL), ec="none"))
x = 0
for gname, gcols in groups:
    ax.text(x + len(gcols) / 2, -0.7, gname, ha="center", va="center", fontsize=9, fontweight="bold")
    ax.plot([x, x], [0, nrow], color="#999", lw=1.4); x += len(gcols)
ax.plot([x, x], [0, nrow], color="#999", lw=1.4)
for jc, col in enumerate(colnames):
    ax.text(jc + 0.5, -0.15, head[col], ha="center", va="center", fontsize=7)
save_fig(fig, "fig4_integrated_evidence_map", fig4_src,
    "Figure 4. Integrated evidence across the seed-excluded top-25 union (rows; \u2605 = observed BTLA "
    "seed; left colour bar = shared/green, GREmLN/blue, GENIE3/orange). Columns are grouped into model "
    "prioritisation (ranks), functional validation (CRISPRi native fraction as a blue gradient, "
    "direction +/\u2212/~, on-target QC \u2713/\u2717), independent multi-omics (protein/phospho/co-IP; "
    "filled = present), derived/contextual multi-omics (transcript/TF-activity/kinase/BIONIC), and "
    "literature (Paperclip tier, purple gradient). No cells are combined into a single opaque score.")
plt.show()

## 6. Figure 5 — coverage and universe audit

In [ ]:
fig5_src = fig1_src.copy()
fig, (axF, axH) = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [1.25, 1]})
funnel = [("BTLA panel", n_panel), ("GREmLN-avail seeds", n_gre), ("GENIE3-reach seeds", n_gen),
          ("common seeds", n_cs), ("common regulator universe", n_univ), ("bona-fide TF universe", n_bona),
          ("seed-excluded primary", n_prim), ("top-25 union", n_union), ("in CRISPRi screen", n_screen),
          ("on-target QC passed", n_qc)]
labels = [f[0] for f in funnel]; vals = [f[1] for f in funnel]; yy = np.arange(len(funnel))[::-1]
axF.barh(yy, vals, color="#6BAED6", edgecolor="#2171B5")
for i, v in zip(yy, vals): axF.text(v + max(vals) * 0.01, i, str(v), va="center", fontsize=8)
axF.set_yticks(yy); axF.set_yticklabels(labels, fontsize=8); axF.set_xlabel("count (genes / seeds / TFs)")
axF.set_title("A  Analysis-universe funnel", loc="left", fontsize=11)
crn = D["crispri_all_screen_sensitivity"]
for m, col in (("GREmLN", GRE), ("GENIE3", GEN)):
    vv = crn[crn.model == m]["n_predicted_native"].astype(float).dropna()
    axH.hist(vv, bins=range(0, int(crn["n_predicted_native"].max()) + 2), alpha=0.6, color=col, label=m)
axH.set_xlabel("predicted BTLA targets per TF (native)"); axH.set_ylabel("number of top-25 TFs")
axH.set_title("B  Predicted-target-count distribution", loc="left", fontsize=11); axH.legend(fontsize=8)
save_fig(fig, "fig5_coverage_audit", fig5_src,
    "Figure 5. Coverage and analysis-universe audit. (A) Funnel of exact counts from the 249-gene BTLA "
    "panel through GREmLN-available and GENIE3-reachable seeds, the identical common seed set, the "
    "common regulator universe, the bona-fide TF universe, the seed-excluded primary universe, the "
    "top-25 union, and CRISPRi in-screen / on-target-QC-passed candidates (a funnel, not a proportional "
    "Venn). (B) Distribution of predicted BTLA-target counts per top-25 TF for each model.")
plt.show()

## 7. Figure 6 — evidence-tiered shortlist

In [ ]:
mo = D["multiomics_candidate_evidence"].set_index("TF"); pc = D["paperclip_candidate_evidence"].set_index("TF")
sl = ci.copy()
def _corrob(tf):
    ind = bool(mo.loc[tf, "independent_orthogonal_support"]) if tf in mo.index else False
    lit = str(pc.loc[tf, "paperclip_evidence_tier"]).lower() in ("strong", "moderate") if tf in pc.index else False
    return int(ind) + int(lit)
sl["functional"] = sl["frac_native_8hr"].astype(float).fillna(0.0)
sl["corroboration"] = [ _corrob(t) for t in sl["TF"] ]
fig6_src = sl[["TF", "status", "is_BTLA_vs_TCR_seed", "functional", "corroboration", "crispri_status"]].copy()
print("source data for Figure 6:"); display(fig6_src)

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 7))
rng = np.random.default_rng(0); texts = []
for _, r in sl.iterrows():
    x = r["functional"]; y = r["corroboration"] + rng.uniform(-0.12, 0.12)
    col = STATUS_COLOR.get(r["status"], NEUTRAL)
    seed = bool(r["is_BTLA_vs_TCR_seed"])
    ax.scatter(x, y, s=70, color=col, edgecolor=(SEED_EDGE if seed else "white"),
               linewidth=(1.8 if seed else 0.8), marker=("D" if seed else "o"), zorder=3)
    if r["TF"] in PRINCIPAL:
        texts.append(ax.text(x, y, r["TF"], fontsize=8))
try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, only_move={"text": "xy"}, arrowprops=dict(arrowstyle="-", color="#999", lw=0.5))
except Exception as e:
    print("adjustText skipped:", e)
ax.axvline(0.001, color="#bbb", ls="--", lw=0.8)
ax.set_xlabel("functional target-response evidence (native CRISPRi fraction, Stim8hr)")
ax.set_ylabel("independent + literature corroboration (0\u20132)")
ax.set_yticks([0, 1, 2]); ax.set_title("Evidence-tiered shortlist of BTLA-response TF candidates")
ax.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor=SHARED, markersize=10, label="shared"),
                   Line2D([0], [0], marker="o", color="w", markerfacecolor=GRE, markersize=10, label="GREmLN-specific"),
                   Line2D([0], [0], marker="o", color="w", markerfacecolor=GEN, markersize=10, label="GENIE3-specific"),
                   Line2D([0], [0], marker="D", color="w", markerfacecolor="#999", markeredgecolor="k", markersize=10, label="observed seed TF")],
          loc="upper right", fontsize=8)
save_fig(fig, "fig6_candidate_shortlist", fig6_src,
    "Figure 6. Evidence-tiered shortlist. Each candidate is placed by functional target-response "
    "evidence (x, native CRISPRi fraction) and independent+literature corroboration (y, count 0\u20132: "
    "independent multi-omics and strong/moderate Paperclip). Colour encodes shared (green), "
    "GREmLN-specific (blue) or GENIE3-specific (orange); diamonds outlined in black are observed BTLA "
    "seed TFs (context, not discoveries). Only principal candidates are labelled. No composite score is "
    "used; axes are the raw evidence dimensions.")
plt.show()

## 8. Table 1 — model and benchmark specification

In [ ]:
t1 = D["model_benchmark_specification"].rename(columns={
    "characteristic": "Characteristic", "implication_for_comparison": "Implication for comparison"})
t1 = t1[["Characteristic", "GENIE3", "GREmLN", "Implication for comparison"]]
print("source data for Table 1:"); display(t1)
out1 = export_table(t1, "table1_model_benchmark_specification",
    "Table 1. Model and benchmark specification",
    ["GENIE3: tree-ensemble co-expression inference on donor-ComBat log1p-CPM CD4 NTC cells.",
     "GREmLN: graph-aware single-cell foundation model, frozen (zero-shot), using the masked GENIE3 graph as prior.",
     "Both models share the identical common seed set and common TF universe; scores are unsigned and compared by rank."])
show_table(out1, "Table 1. Model and benchmark specification")

## 9. Table 2 — primary benchmark results

In [ ]:
ms = D["crispri_model_summary"].set_index("model"); prim_b = primary[primary.is_bona_fide_tf]
rho = spearmanr(prim_b.gremln_dense_rank, prim_b.genie3_dense_rank)[0]
union = D["top25_union_primary"]; shared = int((union.status == "shared").sum())
def _row(metric, gr, ge, diff="", ci="", interp=""):
    return {"Metric": metric, "GREmLN": gr, "GENIE3": ge, "Difference": diff, "95% CI": ci, "Interpretation": interp}
gm, gn = ms.loc["GREmLN"], ms.loc["GENIE3"]
d_mean = gm["mean_frac_native_8hr"] - gn["mean_frac_native_8hr"]
rows = [
    _row("Common BTLA seeds (identical)", n_common_seed, n_common_seed, "0", "", "same seed set (fair)"),
    _row("Bona-fide common TF universe", int(au.is_bona_fide_tf.sum()), int(au.is_bona_fide_tf.sum()), "0", "", "same universe"),
    _row("Spearman rank correlation", "%.3f" % rho, "%.3f" % rho, "", "", "moderate concordance"),
    _row("Top-25 overlap (shared)", shared, shared, "", "", "of 25 each"),
    _row("Present in CRISPRi screen", int(gm["present_in_screen"]), int(gn["present_in_screen"]), "", "", "coverage"),
    _row("Passing on-target QC", int(gm["qc_passed"]), int(gn["qc_passed"]), "", "", "usable perturbations"),
    _row("Included in primary comparison", int(gm["n_in_primary"]), int(gn["n_in_primary"]), "", "", "QC-passed, in-screen"),
    _row("Mean target-response fraction (QC)", "%.3f" % gm["mean_frac_native_8hr"], "%.3f" % gn["mean_frac_native_8hr"],
         "%+.3f" % d_mean, "[%.3f, %.3f] / [%.3f, %.3f]" % (gm["ci_lo"], gm["ci_hi"], gn["ci_lo"], gn["ci_hi"]),
         "within 0.020 margin" if abs(d_mean) < 0.02 else "exceeds margin"),
    _row("Median target-response fraction (QC)", "%.3f" % gm["median_frac_native_8hr"], "%.3f" % gn["median_frac_native_8hr"], "", "", ""),
    _row("BTLA-concordant candidates", int(gm["dir_BTLA_concordant"]), int(gn["dir_BTLA_concordant"]), "", "", "supportive direction"),
    _row("Anti-concordant candidates", int(gm["dir_anti_concordant"]), int(gn["dir_anti_concordant"]), "", "", "opposite direction (not auto-contradictory)"),
    _row("Mixed-direction candidates", int(gm["dir_mixed"]), int(gn["dir_mixed"]), "", "", ""),
    _row("No detected response", int(gm["dir_no_response"]), int(gn["dir_no_response"]), "", "", ""),
    _row("Matched budget: top-5 mean", "%.3f" % gm["mean_frac_top5_8hr"], "%.3f" % gn["mean_frac_top5_8hr"], "", "", "sensitivity"),
    _row("Matched budget: top-10 mean", "%.3f" % gm["mean_frac_top10_8hr"], "%.3f" % gn["mean_frac_top10_8hr"], "", "", "sensitivity"),
    _row("All-screen native mean (sensitivity)", "%.3f" % gm["mean_frac_all_screen_native_8hr"], "%.3f" % gn["mean_frac_all_screen_native_8hr"], "", "", "includes non-QC"),
]
t2 = pd.DataFrame(rows)
print("source data for Table 2:"); display(t2)
out2 = export_table(t2, "table2_primary_benchmark_results", "Table 2. Primary benchmark results",
    ["Target-response fraction = confirmed / predicted BTLA-DEG targets in the Stim8hr CRISPRi arm (adj p < 0.05).",
     "95% CI from 5,000 bootstrap resamples of QC-passed candidates. Superiority margin = 0.020 (pre-specified).",
     "Both scores are unsigned; direction is reported separately and anti-concordance is not automatically contradictory.",
     "En dash (\u2013) = not applicable."])
show_table(out2, "Table 2. Primary benchmark results")

## 10. Table 3 — top-25 candidate regulators (GREmLN vs GENIE3)

Two ranked lists side by side over the **pre-specified pySCENIC 334-candidate** universe, seed-excluded
primary rankings, unchanged from the canonical package. Shared top-25 candidates are bolded, marked
superscript **S**, and shaded pale green in both columns.

In [ ]:
# ---- fixed shading + shared-marker ----
SHARE_BG = "#E3F1E1"                    # subtle pale green (shared candidates only)
SUP_S = "\u02e2"; SIGMA = "\u03a3"; DASH = "\u2014"   # superscript s, capital sigma, em dash
def _mbold(sym, suffix=""):             # image-only mathtext: bold symbol + superscript S
    return r"$\mathbf{%s}^{\mathrm{S}}$%s" % (sym, suffix)

pr = D["primary_rankings_common_universe"].copy()          # seed-excluded primary (unchanged)
assert not pr["is_BTLA_vs_TCR_seed"].any(), "seeds present in primary ranking"
assert "genie3_n_seed_targets" in pr.columns, "canonical package predates the GENIE3 seed-target count; re-run nb03"
sc = pr.set_index("TF")
u25 = D["top25_union_primary"].set_index("TF")
gr_members = set(u25.index[u25.in_gremln_top25]); g3_members = set(u25.index[u25.in_genie3_top25])
# Membership is taken UNCHANGED from the canonical package. GREmLN ranks by dense seed count with a
# continuous CSLS seed-sum tie-break (this also fixes the top-25 boundary deterministically). GENIE3
# ranks by summed edge weight (near-strict); residual ties are ordered alphabetically.
GR_SORT = (["gremln_dense_rank", "gremln_csls_seed_sum"], [True, False])
assert gr_members == set(pr.sort_values(GR_SORT[0], ascending=GR_SORT[1]).head(25).TF), "GREmLN top-25 differs from canonical"
assert g3_members == set(pr.sort_values("genie3_dense_rank").head(25).TF), "GENIE3 top-25 differs from canonical"
assert len(gr_members) == 25 and len(g3_members) == 25, "top-25 lists are not exactly 25"
gr_ord = pr[pr.TF.isin(gr_members)].sort_values(GR_SORT[0], ascending=GR_SORT[1]).reset_index(drop=True)
g3_ord = pr[pr.TF.isin(g3_members)].sort_values(["genie3_dense_rank", "TF"]).reset_index(drop=True)
shared = gr_members & g3_members
print("Table 3 basis: 25 GREmLN + 25 GENIE3; shared =", len(shared), "->", sorted(shared))

rows, cbg, bold, disp = [], {}, {}, {}
GR_COL, G3_COL = 1, 3
gr_pos = {tf: k + 1 for k, tf in enumerate(gr_ord.TF)}   # displayed ordinal position in the GREmLN half
g3_pos = {tf: k + 1 for k, tf in enumerate(g3_ord.TF)}   # displayed ordinal position in the GENIE3 half
for i in range(25):
    g = gr_ord.TF.iloc[i]; h = g3_ord.TF.iloc[i]
    gs = int(sc.loc[g, "gremln_score"]); gcs = float(sc.loc[g, "gremln_csls_seed_sum"])
    hn = int(sc.loc[h, "genie3_n_seed_targets"]); hw = float(sc.loc[h, "genie3_score"])
    if g in shared:                          # bracket shows this candidate's ORDINAL POSITION in the GENIE3 half
        gr_cell = f"{g}{SUP_S} (G3 {g3_pos[g]})"
        disp[(i, GR_COL)] = _mbold(g, f" (G3 {g3_pos[g]})"); cbg[(i, GR_COL)] = SHARE_BG; bold[(i, GR_COL)] = True
    else:
        gr_cell = g
    if h in shared:                          # bracket shows this candidate's ORDINAL POSITION in the GREmLN half
        g3_cell = f"{h}{SUP_S} (GR {gr_pos[h]})"
        disp[(i, G3_COL)] = _mbold(h, f" (GR {gr_pos[h]})"); cbg[(i, G3_COL)] = SHARE_BG; bold[(i, G3_COL)] = True
    else:
        g3_cell = h
    rows.append({"Position": i + 1, "GREmLN candidate": gr_cell,
                 "GREmLN selection signal": f"{gs} seeds; {SIGMA}CSLS = {gcs:.3f}",
                 "GENIE3 candidate": g3_cell, "GENIE3 selection signal": f"{hn}; {SIGMA}w = {hw:.3f}"})
t3 = pd.DataFrame(rows)
print("source data for Table 3:"); display(t3)
t3_notes = [
    "S, shared between both model top-25 lists; brackets give ordinal position in the other list.",
    "BTLA panel genes were excluded from the primary candidate ranking.",
    "GREmLN uses BTLA seed-neighbour count with summed CSLS as tie-break; GENIE3 uses summed outgoing "
    "edge weight to BTLA seed targets.",
    "Within equal seed-neighbour counts, higher (less negative) summed CSLS ranks higher.",
    "The two model-specific selection signals are not directly comparable."]
export_table(t3, "table3_top25_rankings", "Table 3. Top-25 candidate regulators prioritised by GREmLN and GENIE3",
             t3_notes, cell_bg=cbg, bold_cells=bold, disp=disp, width=1150)

# ---- assertion: every bracketed cross-model position matches the visible position in the other half ----
def _sym(cell): return str(cell).split(SUP_S)[0].split(" (")[0]
gr_seen = {_sym(r["GREmLN candidate"]): r["Position"] for _, r in t3.iterrows()}
g3_seen = {_sym(r["GENIE3 candidate"]): r["Position"] for _, r in t3.iterrows()}
for tf in shared:
    assert g3_pos[tf] == g3_seen[tf] and gr_pos[tf] == gr_seen[tf], f"position map disagrees with table for {tf}"
    gr_cell_tf = t3.loc[t3["GREmLN candidate"].map(_sym) == tf, "GREmLN candidate"].iloc[0]
    g3_cell_tf = t3.loc[t3["GENIE3 candidate"].map(_sym) == tf, "GENIE3 candidate"].iloc[0]
    assert f"(G3 {g3_seen[tf]})" in gr_cell_tf, f"GREmLN-half bracket wrong for {tf}: {gr_cell_tf}"
    assert f"(GR {gr_seen[tf]})" in g3_cell_tf, f"GENIE3-half bracket wrong for {tf}: {g3_cell_tf}"
print("cross-model ordinal-position brackets verified for", len(shared), "shared candidates")

### Supplementary Table S1 — full 43-candidate union (ranks and raw scores)

In [ ]:
sup = u25.reset_index().merge(
    pr[["TF", "gremln_score", "gremln_csls_seed_sum", "genie3_score", "genie3_n_seed_targets"]], on="TF", how="left")
s1 = pd.DataFrame({
    "Candidate": sup.TF, "Model status": sup.status.str.replace("_", " "),
    "GREmLN dense rank": sup.gremln_dense_rank.astype(int),
    "GREmLN seeds (top-100)": sup.gremln_score.astype(int),
    "GREmLN CSLS seed-sum": sup.gremln_csls_seed_sum.round(3),
    "GENIE3 dense rank": sup.genie3_dense_rank.astype(int),
    "GENIE3 linked seed targets": sup.genie3_n_seed_targets.astype(int),
    "GENIE3 summed weight": sup.genie3_score.round(3),
}).sort_values(["GREmLN dense rank", "GREmLN CSLS seed-sum", "GENIE3 dense rank"],
               ascending=[True, False, True]).reset_index(drop=True)
print("Supplementary S1 rows:", len(s1)); display(s1)
export_table(s1, "supplementary_table_s1_full_top25_union",
    "Supplementary Table S1. Full top-25 union of candidate regulators (43 candidates)",
    ["All candidates entering either model's top-25 over the pySCENIC 334-candidate universe.",
     "Model status: shared / GREmLN-specific / GENIE3-specific. Ranks are dense/tie-aware.",
     "GREmLN and GENIE3 raw scores are on different scales and are not directly comparable."], width=1150)

## 11. Table 4 — evidence across each model's top-25

Two matched sub-tables (**4A** GREmLN, **4B** GENIE3) with five columns: `Position`, `Candidate
regulator`, `CRISPRi evidence`, `Literature evidence`, `BTLA experimental evidence`. Every evidence
label is generated from the canonical long-form evidence table (`multiomics_long_evidence.csv`) via an
explicit provenance audit — no labels are hand-typed. For shared candidates the **model-specific**
CRISPRi target set is used in each table. `BTLA experimental evidence` covers the BTLA-engagement
contrasts (**BTLA+TCR vs TCR** and **BTLA+TCR vs UC**); the generic TCR-activation contrast (TCR vs
UC) is excluded and retained only in Supplementary S2.

### Evidence provenance audit (long-form → displayed labels)

In [ ]:
# ---- unicode marks + colours ----
DAGGER = "\u2020"; ARR_UP = "\u2191"; ARR_DN = "\u2193"
PG = SHARE_BG; GREY = "#EEEEEE"                          # pale green (strongest) / light grey (none/untested)

crx = D["crispri_all_screen_sensitivity"].set_index(["TF", "model"])
pcv = D["paperclip_candidate_evidence"].set_index("TF")
lf = D["multiomics_long_evidence"].copy()

# BTLA experimental evidence covers the BTLA-engagement contrasts (isolate/attribute the BTLA effect);
# the generic TCR-activation contrast (TCR_vs_UC) is excluded and retained only in Supplementary S2.
BTLA_CONTRASTS = ("BTLA_TCR_vs_TCR", "BTLA_TCR_vs_UC"); SUPPORTED = ("moderate", "strong")
LAYER_LABEL = {"proteomics": "Protein" + DAGGER, "phosphoproteomics": "Phosphosite" + DAGGER,
               "coIP": "Co-IP" + DAGGER, "transcriptomics": "Transcript",
               "tf_activity": "Inferred regulon activity", "bionic_gnn": "BIONIC"}
LAYER_ORDER = ["proteomics", "phosphoproteomics", "coIP", "transcriptomics", "tf_activity", "bionic_gnn"]
ORTHO = {"proteomics", "phosphoproteomics", "coIP"}      # orthogonal molecular (dagger; pale green)

def _method(layer):
    if layer == "tf_activity": return "decoupleR ULM (inferred regulon activity)"
    if layer == "bionic_gnn": return "BIONIC GNN network module"
    return "direct measurement"

# provenance audit: one row per candidate-evidence observation, with the inclusion decision recorded
audit_rows = []
for _, r in lf.iterrows():
    layer = str(r["evidence_layer"]); gene = str(r["gene"]); contrast = str(r["contrast"])
    sup = str(r["support_level"]).lower(); direction = str(r["direction"])
    shown = LAYER_LABEL.get(layer); keep = True
    if shown is None:
        keep, reason = False, "layer not shown in main table (" + layer + ")"
    elif sup not in SUPPORTED:
        keep, reason = False, "not significant (support_level=" + sup + ")"
    elif layer == "bionic_gnn":
        reason = "included: BIONIC network module (contrast-free)"
    elif contrast not in BTLA_CONTRASTS:
        keep, reason = False, "contrast not BTLA-specific (" + contrast + "); retained in Supplementary S2"
    else:
        reason = "included: significant at " + contrast
    disp_label = shown if shown else EN
    if keep and layer == "tf_activity":
        disp_label = shown + (" " + ARR_UP if direction == "up" else " " + ARR_DN if direction == "down" else "")
    audit_rows.append({
        "candidate": gene, "model": (u25.loc[gene, "status"] if gene in u25.index else "not_top25"),
        "evidence_layer": layer, "contrast": contrast, "timepoint": r["timepoint"], "condition": r["condition"],
        "direction": direction,
        "effect_or_score": (r["tf_activity_score"] if layer == "tf_activity" else r["effect_size"]),
        "adjusted_p": r["adjusted_p"], "inference_method": _method(layer),
        "regulon_resource": (Path(str(r["source_file"])).name if layer == "tf_activity" else EN),
        "source_file": r["source_file"], "independence": r["independent_or_derived"],
        "displayed_label": (disp_label if keep else EN), "included": keep, "inclusion_reason": reason})
audit = pd.DataFrame(audit_rows)
print("evidence provenance audit rows:", len(audit), "| included:", int(audit["included"].sum()),
      "| candidates with >=1 included:", audit.loc[audit["included"], "candidate"].nunique())
display(audit[audit["included"]].sort_values(["candidate", "evidence_layer", "timepoint"]).reset_index(drop=True))

In [ ]:
# ---- displayed-cell builders (all derived from the audit / canonical CRISPRi) ----
def crispri_cell(tf, model):
    key = (tf, model)
    if key not in crx.index or not bool(crx.loc[key, "in_screen_8hr"]): return "Not in screen"
    r = crx.loc[key]
    if r["ontarget_qc_pass"] != True: return "Failed on-target QC"
    npred = int(r["n_predicted_native"]) if pd.notna(r["n_predicted_native"]) else 0
    nconf = int(r["n_confirmed_native_8hr"]) if pd.notna(r["n_confirmed_native_8hr"]) else 0
    nd = f" ({nconf}/{npred})"; d = str(r["response_direction"])
    return {"BTLA_concordant": f"Detected {DASH} BTLA-concordant" + nd,
            "anti_concordant": f"Detected {DASH} BTLA-anti-concordant" + nd,
            "mixed": f"Detected {DASH} mixed" + nd}.get(d, "No detected response" + nd)

def lit_cell(tf):
    tier = str(pcv.loc[tf, "paperclip_evidence_tier"]).lower() if tf in pcv.index else "none"
    return {"strong": "Strong", "moderate": "Moderate", "weak": "Weak"}.get(tier, "None")

def btla_exp_evidence(tf):
    sub = audit[(audit["candidate"] == tf) & (audit["included"])]
    labels = []
    for layer in LAYER_ORDER:
        ls = sub[sub["evidence_layer"] == layer]
        if not len(ls): continue
        if layer == "tf_activity":
            prim = ls[ls["contrast"] == "BTLA_TCR_vs_TCR"]      # arrow follows the primary contrast if present
            dirs = set((prim if len(prim) else ls)["direction"])
            arrow = (" " + ARR_UP) if dirs == {"up"} else (" " + ARR_DN) if dirs == {"down"} else ""
            labels.append(LAYER_LABEL[layer] + arrow)
        else:
            labels.append(LAYER_LABEL[layer])
    return "; ".join(labels) if labels else "None"

def has_ortho(tf):
    sub = audit[(audit["candidate"] == tf) & (audit["included"])]
    return bool(len(sub[sub["evidence_layer"].isin(ORTHO)]))

CAND_COL, CRIS_COL, LIT_COL, EXP_COL = 1, 2, 3, 4
def build_evidence(model, order_df):
    rows, cbg, bold, disp = [], {}, {}, {}
    for i in range(25):
        tf = order_df.TF.iloc[i]
        if tf in shared:
            cand = f"{tf}{SUP_S}"; disp[(i, CAND_COL)] = _mbold(tf); cbg[(i, CAND_COL)] = PG; bold[(i, CAND_COL)] = True
        else:
            cand = tf
        cris = crispri_cell(tf, model)
        if cris.startswith("Detected"): cbg[(i, CRIS_COL)] = PG                        # any detected response
        elif cris in ("Not in screen", "Failed on-target QC"): cbg[(i, CRIS_COL)] = GREY
        lab = lit_cell(tf)
        if lab == "Strong": cbg[(i, LIT_COL)] = PG
        elif lab == "None": cbg[(i, LIT_COL)] = GREY                                   # moderate/weak -> white
        exp = btla_exp_evidence(tf)
        if has_ortho(tf): cbg[(i, EXP_COL)] = PG                                        # orthogonal molecular
        elif exp == "None": cbg[(i, EXP_COL)] = GREY                                    # transcript/inferred/BIONIC -> white
        rows.append({"Position": i + 1, "Candidate regulator": cand, "CRISPRi evidence": cris,
                     "Literature evidence": lab, "BTLA experimental evidence": exp})
    return pd.DataFrame(rows), cbg, bold, disp

T4_NOTES = [
    "S, shared between the GREmLN and GENIE3 top-25 lists.",
    "Candidate regulators were restricted to the pre-specified pySCENIC human transcription-factor/regulator list.",
    "CRISPRi evidence is model-specific because each model can nominate a different target set for the same candidate.",
    "BTLA-concordant and BTLA-anti-concordant describe response direction and are not automatically supportive or contradictory.",
    "Transcript and inferred regulon activity are corroborating evidence from the BTLA transcriptomic study, not independent validation.",
    "BIONIC is a derived network annotation.",
    DAGGER + ", orthogonal molecular evidence from protein, phosphosite or co-IP data.",
    "Paperclip literature evidence was not used to generate the model rankings.",
    "Inferred regulon activity: decoupleR ULM on the BTLA regulon-activity table (TFactivitiesALL.xlsx), "
    "BTLA-engagement contrasts (BTLA+TCR vs TCR and BTLA+TCR vs UC), 1" + EN + "24 h; " + ARR_UP + "/" + ARR_DN
    + " = increased/decreased inferred activity (following BTLA+TCR vs TCR where available).",
    "Pale green marks the strongest evidence: a detected CRISPRi response (any direction), strong literature, or "
    "orthogonal molecular evidence. Light grey marks not-in-screen, failed QC or no evidence; other cells are white."]

t4a, a_bg, a_bold, a_disp = build_evidence("GREmLN", gr_ord)
t4b, b_bg, b_bold, b_disp = build_evidence("GENIE3", g3_ord)
# identical column widths across 4A/4B so the two evidence profiles are visually comparable
COLW4 = [min(34, max(9, max(len(str(cn)),
             int(t4a[cn].astype(str).map(len).max()), int(t4b[cn].astype(str).map(len).max()))))
         for cn in t4a.columns]
DMAP = {"None": EN}                                        # DOCX-only: replace repeated "None" with an en-dash
print("source data for Table 4A (GREmLN):"); display(t4a)
t4a_out = export_table(t4a, "table4a_gremln_top25_evidence",
    "Table 4A. Evidence across GREmLN's top-25 candidate regulators",
    T4_NOTES, cell_bg=a_bg, bold_cells=a_bold, disp=a_disp, width=1250, col_chars=COLW4, zebra=False, docx_map=DMAP)

In [ ]:
print("source data for Table 4B (GENIE3):"); display(t4b)
t4b_out = export_table(t4b, "table4b_genie3_top25_evidence",
    "Table 4B. Evidence across GENIE3's top-25 candidate regulators",
    T4_NOTES, cell_bg=b_bg, bold_cells=b_bold, disp=b_disp, width=1250, col_chars=COLW4, zebra=False, docx_map=DMAP)

### Cross-model evidence summary (descriptive; no weighted score or winner)

In [ ]:
def summarise(model, order_df):
    d = dict(scr=0, qc=0, use=0, det=0, conc=0, anti=0, lit=0, exp=0, ortho=0)
    for tf in list(order_df.TF):
        key = (tf, model)
        if key in crx.index and bool(crx.loc[key, "in_screen_8hr"]):
            d["scr"] += 1
            if crx.loc[key, "ontarget_qc_pass"] == True:
                d["qc"] += 1; d["use"] += 1
                rd = str(crx.loc[key, "response_direction"])
                if rd in ("BTLA_concordant", "anti_concordant", "mixed"): d["det"] += 1
                if rd == "BTLA_concordant": d["conc"] += 1
                if rd == "anti_concordant": d["anti"] += 1
        if lit_cell(tf) in ("Strong", "Moderate"): d["lit"] += 1
        if btla_exp_evidence(tf) != "None": d["exp"] += 1
        if has_ortho(tf): d["ortho"] += 1
    return d
sg, se = summarise("GREmLN", gr_ord), summarise("GENIE3", g3_ord)
summ = pd.DataFrame([
    {"Evidence summary": "Present in CRISPRi screen", "GREmLN top 25": f"{sg['scr']}/25", "GENIE3 top 25": f"{se['scr']}/25"},
    {"Evidence summary": "Passing on-target QC", "GREmLN top 25": f"{sg['qc']}/25", "GENIE3 top 25": f"{se['qc']}/25"},
    {"Evidence summary": "Target response detected", "GREmLN top 25": f"{sg['det']}/{sg['use']} usable", "GENIE3 top 25": f"{se['det']}/{se['use']} usable"},
    {"Evidence summary": "BTLA-concordant response", "GREmLN top 25": f"{sg['conc']}/{sg['use']} usable", "GENIE3 top 25": f"{se['conc']}/{se['use']} usable"},
    {"Evidence summary": "BTLA-anti-concordant response", "GREmLN top 25": f"{sg['anti']}/{sg['use']} usable", "GENIE3 top 25": f"{se['anti']}/{se['use']} usable"},
    {"Evidence summary": "Strong or moderate literature", "GREmLN top 25": f"{sg['lit']}/25", "GENIE3 top 25": f"{se['lit']}/25"},
    {"Evidence summary": "Any BTLA experimental evidence", "GREmLN top 25": f"{sg['exp']}/25", "GENIE3 top 25": f"{se['exp']}/25"},
    {"Evidence summary": "Orthogonal molecular evidence" + DAGGER, "GREmLN top 25": f"{sg['ortho']}/25", "GENIE3 top 25": f"{se['ortho']}/25"}])
print("Cross-model evidence summary:"); display(summ)
export_table(summ, "table4_evidence_summary", "Table 4. Cross-model evidence summary (GREmLN vs GENIE3 top-25)",
    ["Descriptive comparison of evidence coverage across each model's top-25; no weighted score or winner is implied.",
     "Detected / concordant / anti-concordant responses use QC-passed candidates as the denominator (usable).",
     DAGGER + " Orthogonal molecular evidence = protein, phosphosite or co-IP; BTLA experimental evidence also "
     "includes transcript, inferred regulon activity and BIONIC."],
    width=780)

### Supplementary Table S2 — full long-form candidate evidence

The complete per-observation evidence audit (all layers, contrasts and timepoints for every top-25
union candidate, with the inclusion decision for the main table). Data-only export (CSV/XLSX/HTML).

In [ ]:
s2 = audit.sort_values(["candidate", "evidence_layer", "contrast", "timepoint"]).reset_index(drop=True)
print("Supplementary S2 (long-form) rows:", len(s2), "| candidates:", s2.candidate.nunique()); display(s2.head(20))
S2 = "supplementary_table_s2_full_candidate_evidence"
s2.to_csv(TAB / (S2 + ".csv"), index=False)
try: s2.to_excel(TAB / (S2 + ".xlsx"), index=False)
except Exception as e: print("xlsx skipped:", e)
(TAB / (S2 + ".html")).write_text("<h3>Supplementary Table S2. Full long-form candidate evidence</h3>"
    + s2.to_html(index=False, border=0))
(CAP / (S2 + "_caption.txt")).write_text(
    "Supplementary Table S2. Full long-form candidate evidence." + chr(10)
    + "One row per candidate-evidence observation across all layers, contrasts and timepoints; "
      "'included' and 'inclusion_reason' record whether the observation contributes to the main Table 4 "
      "BTLA experimental-evidence column (significant at a BTLA-engagement contrast, BTLA+TCR vs TCR or "
      "BTLA+TCR vs UC, or a BIONIC module)." + chr(10))
print("supplementary S2 written (csv/xlsx/html + caption)")

### Table 3/4 quality checks

In [ ]:
tchecks = {}
tchecks["table3_25_rows"] = len(t3) == 25
tchecks["table4a_25_rows"] = len(t4a) == 25
tchecks["table4b_25_rows"] = len(t4b) == 25
tchecks["rankings_match_canonical"] = (gr_members == set(pr.sort_values(GR_SORT[0], ascending=GR_SORT[1]).head(25).TF)
    and g3_members == set(pr.sort_values("genie3_dense_rank").head(25).TF))
_sh4a = set(t4a.loc[t4a["Candidate regulator"].str.contains(SUP_S), "Candidate regulator"].str.replace(SUP_S, "", regex=False))
_sh4b = set(t4b.loc[t4b["Candidate regulator"].str.contains(SUP_S), "Candidate regulator"].str.replace(SUP_S, "", regex=False))
tchecks["shared_consistent"] = (_sh4a == shared) and (_sh4b == shared)
tchecks["shared_model_specific_crispri"] = all(((tf, "GREmLN") in crx.index and (tf, "GENIE3") in crx.index) for tf in shared)
tchecks["seeds_absent_primary"] = not pr["is_BTLA_vs_TCR_seed"].any()
# failed-QC / not-in-screen must never carry a numeric fraction "(n/N)"
_nofrac = pd.concat([t4a["CRISPRi evidence"], t4b["CRISPRi evidence"]])
tchecks["absent_failed_no_fraction"] = not any(("(" in v) for v in _nofrac
    if v.startswith(("Not in screen", "Failed on-target QC")))
# every displayed BTLA experimental-evidence label traces to >=1 included canonical source row
_inc = set(audit.loc[audit["included"], "candidate"])
tchecks["exp_labels_traceable"] = all(
    (tf in _inc) for t4 in (t4a, t4b)
    for tf, ev in zip(t4["Candidate regulator"].str.replace(SUP_S, "", regex=False), t4["BTLA experimental evidence"])
    if ev != "None")
# every included inferred-regulon-activity row has verified direction/contrast/timepoint/method/regulon source
_ta = audit[(audit["included"]) & (audit["evidence_layer"] == "tf_activity")]
tchecks["regulon_activity_provenance"] = bool(len(_ta)) and (
    _ta["direction"].isin(["up", "down"]).all() and _ta["contrast"].isin(BTLA_CONTRASTS).all()
    and _ta["timepoint"].notna().all() and _ta["inference_method"].notna().all()
    and (_ta["regulon_resource"] != EN).all())
# summary panel equals row-level categories
def _rowcounts(t4):
    det = int(t4["CRISPRi evidence"].str.startswith("Detected").sum())
    scr = int((~t4["CRISPRi evidence"].eq("Not in screen")).sum())
    qc = int((~t4["CRISPRi evidence"].isin(["Not in screen", "Failed on-target QC"])).sum())
    conc = int(t4["CRISPRi evidence"].str.startswith("Detected " + DASH + " BTLA-concordant").sum())
    anti = int(t4["CRISPRi evidence"].str.startswith("Detected " + DASH + " BTLA-anti-concordant").sum())
    lit = int(t4["Literature evidence"].isin(["Strong", "Moderate"]).sum())
    exp = int((t4["BTLA experimental evidence"] != "None").sum())
    return dict(det=det, scr=scr, qc=qc, conc=conc, anti=anti, lit=lit, exp=exp)
_ag = _rowcounts(t4a); _bg2 = _rowcounts(t4b)
tchecks["summary_matches_rows"] = all(_ag[k] == sg[k] for k in _ag) and all(_bg2[k] == se[k] for k in _bg2)
# DOCX/XLSX/CSV/HTML carry identical evidence values (csv==xlsx cell-for-cell; html/docx present and value-complete)
import re as _re
for _n, _tag in [("table4a_gremln_top25_evidence", "4A"), ("table4b_genie3_top25_evidence", "4B")]:
    # keep_default_na=False so the explicit label "None" is not coerced to NaN on readback
    _csv = pd.read_csv(TAB / (_n + ".csv"), keep_default_na=False).astype(str)
    _xls = pd.read_excel(TAB / (_n + ".xlsx"), keep_default_na=False).astype(str)
    _htxt = _re.sub("<[^>]+>", " ", (TAB / (_n + ".html")).read_text())
    _vals_in_html = all((v in _htxt) for v in _csv["BTLA experimental evidence"].unique())
    _docx_ok = (TAB / (_n + ".docx")).exists()
    tchecks["exports_identical_" + _tag] = _csv.equals(_xls) and _vals_in_html and _docx_ok
for k, v in tchecks.items(): print(("PASS" if v else "FAIL"), k)
assert all(tchecks.values()), "Table 3/4 checks failed: " + ", ".join(k for k, v in tchecks.items() if not v)
print(chr(10) + "Table 3/4 quality checks passed.")

## 12. Asset index and final quality checks

In [ ]:
figs = [("fig1_study_design", "Figure 1", "Introduction/Methods", "Fair study design; both models share the identical seed set and TF universe."),
        ("fig2_primary_rank_comparison", "Figure 2", "Results", "GREmLN and GENIE3 rankings are moderately correlated; 7/25 shared candidates."),
        ("fig3_crispri_benchmark", "Figure 3", "Results", "Neither model wins CD4 CRISPRi target response beyond the 0.020 margin."),
        ("fig4_integrated_evidence_map", "Figure 4", "Results", "Grouped evidence shows most support is literature/derived, not orthogonal."),
        ("fig5_coverage_audit", "Figure 5", "Results/Methods", "Coverage funnel; GREmLN reach is coverage-limited, not algorithm-limited."),
        ("fig6_candidate_shortlist", "Figure 6", "Results/Discussion", "Few non-seed candidates combine functional and corroborating support.")]
tabs = [("table1_model_benchmark_specification", "Table 1", "Methods", "Specification and the primary fairness limitation."),
        ("table2_primary_benchmark_results", "Table 2", "Results", "Ranking, CRISPRi coverage, primary and matched-budget results."),
        ("table3_top25_rankings", "Table 3", "Results", "Side-by-side top-25 candidate regulators; shared candidates marked S."),
        ("table4a_gremln_top25_evidence", "Table 4A", "Results", "Evidence across GREmLN's top-25 candidate regulators."),
        ("table4b_genie3_top25_evidence", "Table 4B", "Results", "Evidence across GENIE3's top-25 candidate regulators."),
        ("table4_evidence_summary", "Table 4 summary", "Results", "Descriptive evidence-coverage comparison (no winner).")]
supp = [("supplementary_table_s1_full_top25_union", "Supp. Table S1", "Supplement", "Full 43-candidate union with ranks and raw scores."),
        ("supplementary_table_s2_full_candidate_evidence", "Supp. Table S2", "Supplement", "Full 43-candidate evidence across both models.")]

lines = ["# Publication assets index", "",
         "Generated by `notebooks/04_publication_figures_and_tables.ipynb` from the canonical package in",
         "`results/publication_data/`. Figures: pdf/svg/png@600dpi + source_data csv + caption txt.",
         "Tables: csv/xlsx/docx/html + caption txt.", "", "## Figures", ""]
for name, num, sec, msg in figs:
    ok = all((FIG / (name + e)).exists() for e in (".pdf", ".svg", ".png"))
    lines.append("- **%s** (`%s`) - %s - %s - status: %s" % (num, name, sec, msg, "complete" if ok else "MISSING"))
lines += ["", "## Tables", ""]
for name, num, sec, msg in tabs + supp:
    exts = (".csv", ".xlsx", ".html") if name.endswith("s2_full_candidate_evidence") else (".csv", ".xlsx", ".html", ".png", ".pdf", ".svg")
    ok = all((TAB / (name + e)).exists() for e in exts)
    lines.append("- **%s** (`%s`) - %s - %s - status: %s" % (num, name, sec, msg, "complete" if ok else "MISSING"))
lines += ["", "## Outstanding caveats", "",
          "- CRISPRi means differ slightly from the earlier provisional report draft (common-seed + matched-target fix); report to be resynced.",
          "- Held-out seed-TF recovery remains supplementary (excluded from the verdict).",
          "- BTLA-derived assets are gitignored pending data-publication clearance."]
(ASSET / "PUBLICATION_ASSETS_INDEX.md").write_text(chr(10).join(lines) + chr(10))
print("wrote", ASSET / "PUBLICATION_ASSETS_INDEX.md")
display(Markdown(chr(10).join(lines)))

In [ ]:
# visual index: thumbnail of each figure + rendered table
for name, num, sec, msg in figs:
    print("%s  |  %s  |  %s" % (num, name, msg)); display(Image(filename=str(FIG / (name + ".png")), width=680))
for name, num, sec, msg in tabs + supp:
    print("%s  |  %s  |  %s" % (num, name, msg))
    _p = TAB / (name + ".png")
    if _p.exists(): display(Image(filename=str(_p), width=900))
    else: print("(data-only supplementary; see CSV/XLSX/HTML)")

In [ ]:
# ---- final quality checks ----
checks = {}
checks["same_common_seed_set"] = int(seedu.in_common_seed_set.sum()) == n_common_seed and n_common_seed > 0
checks["seeds_absent_from_primary"] = not primary["is_BTLA_vs_TCR_seed"].any()
checks["primary_candidates_classified"] = set(primary.TF).issubset(set(au.TF)) and au.is_bona_fide_tf.notna().all()
checks["absent_failed_not_zero"] = len(crispri[(crispri.validation_status.isin(["not_in_screen", "failed_ontarget_qc"]))
                                               & (crispri.frac_native_8hr.fillna(-1) == 0)]) == 0
checks["model_specific_targets_separate"] = set(crispri.model) == {"GREmLN", "GENIE3"}
# table values match figure/source values
_ms = D["crispri_model_summary"].set_index("model")
_t2mean = float(out2.set_index("Metric").loc["Mean target-response fraction (QC)", "GREmLN"])
checks["table_matches_source"] = abs(_t2mean - round(float(_ms.loc["GREmLN", "mean_frac_native_8hr"]), 3)) < 1e-6
checks["captions_present"] = all((CAP / (n + "_caption.txt")).exists() for n, *_ in figs) and \
                             all((CAP / (n + "_caption.txt")).exists() for n, *_ in tabs)
checks["assets_from_canonical"] = all((FIG / (n + ".png")).exists() for n, *_ in figs) and \
                                  all((TAB / (n + ".csv")).exists() for n, *_ in tabs)
for k, v in checks.items():
    print(("PASS" if v else "FAIL"), k)
assert all(checks.values()), "final quality checks failed: " + ", ".join(k for k, v in checks.items() if not v)
print(chr(10) + "All final quality checks passed. Publication assets are in", ASSET)